In [4]:
import json
from pathlib import Path

results_path = Path("wandb_compute_exps/results.json")

with results_path.open("r") as file:
    results = json.load(file)

In [5]:
list(results.keys())

['knn',
 'linear',
 'dnnc',
 'rerank',
 'sklearn',
 'bert',
 'lora',
 'ptuning',
 'system']

In [7]:
_ = results.pop("system")

In [8]:
for scorer_name, res_config_and_metrics in results.items():
    print(f"scorer_name: {scorer_name} has {len(res_config_and_metrics)} runs")

scorer_name: knn has 5 runs
scorer_name: linear has 1 runs
scorer_name: dnnc has 5 runs
scorer_name: rerank has 10 runs
scorer_name: sklearn has 7 runs
scorer_name: bert has 5 runs
scorer_name: lora has 5 runs
scorer_name: ptuning has 5 runs


In [9]:
results["linear"]

[{'config': {'embedder_config': {'value': {'batch_size': 32,
     'classifier_prompt': None,
     'cluster_prompt': None,
     'default_prompt': None,
     'device': None,
     'freeze': True,
     'model_name': 'mixedbread-ai/mxbai-embed-large-v1',
     'passage_prompt': None,
     'query_prompt': None,
     'similarity_fn_name': 'cosine',
     'sts_prompt': None,
     'tokenizer_config': {'max_length': None,
      'padding': True,
      'truncation': True},
     'trust_remote_code': False,
     'use_cache': True}}},
  'metrics': {'emissions/gpu_power': 15.459576302043818,
   'emissions/emissions': 0.0004282634817614922,
   '_step': 0,
   'scoring_recall': 0.9436427707340618,
   'emissions/cpu_count': 16,
   'emissions/duration': -1746690760.1935968,
   'scoring_precision': 0.9486464853393203,
   'emissions/pue': 1,
   'emissions/energy_consumed': 0.0009710331325834955,
   'emissions/gpu_energy': 0.0003121294163701549,
   'scoring_roc_auc': 0.9993575742642029,
   'emissions/ram_energy

In [29]:
from collections import defaultdict
import numpy as np

metrics_to_aggregate = [
    "emissions/emissions",
    "_runtime",
    "emissions/emissons_rate",
    "emissions/energy_consumed",
    "emissions/gpu_power",
    "emissions/cpu_power",
    "emissions/gpu_energy",
    "emissions/cpu_energy",
    "emissions/ram_energy",
    "emissions/emissions_rate",
    "emissions/energy_consumed",
]

aggregated_metrics = {}
for scorer_name, res_config_and_metrics in results.items():
    print(f"scorer_name: {scorer_name} has {len(res_config_and_metrics)} runs")
    grouped_metrics = defaultdict(list)
    for res in res_config_and_metrics:
        for metric in metrics_to_aggregate:
            metric_val = res["metrics"].get(metric, None)
            if metric_val is not None:
                grouped_metrics[metric].append(metric_val)

    medians = {}
    stds = {}
    for metric_name, metric_values in grouped_metrics.items():
        if metric_values:
            metric_name_ = metric_name.split("/")[-1]
            medians[metric_name_] = np.median(metric_values)
            stds[metric_name_] = np.std(metric_values)

    aggregated_metrics[scorer_name] = {"medians": medians, "stds": stds}

aggregated_metrics


scorer_name: knn has 5 runs
scorer_name: linear has 1 runs
scorer_name: dnnc has 5 runs
scorer_name: rerank has 10 runs
scorer_name: sklearn has 7 runs
scorer_name: bert has 5 runs
scorer_name: lora has 5 runs
scorer_name: ptuning has 5 runs


{'knn': {'medians': {'emissions': np.float64(8.534494756430144e-06),
   '_runtime': np.float64(1.280654019),
   'energy_consumed': np.float64(1.935088451685712e-05),
   'gpu_power': np.float64(67.44074098616595),
   'cpu_power': np.float64(27.0),
   'gpu_energy': np.float64(1.4050844574065025e-05),
   'cpu_energy': np.float64(4.339880399638785e-06),
   'ram_energy': np.float64(9.04411058785016e-07),
   'emissions_rate': np.float64(1.2260047910820804e-05)},
  'stds': {'emissions': np.float64(0.0001725836794878742),
   '_runtime': np.float64(14.085032674620392),
   'energy_consumed': np.float64(0.0003913116062023407),
   'gpu_power': np.float64(18.677803847970853),
   'cpu_power': np.float64(0.0),
   'gpu_energy': np.float64(0.00026359246974820436),
   'cpu_energy': np.float64(0.00010567229637723178),
   'ram_energy': np.float64(2.2051801472265185e-05),
   'emissions_rate': np.float64(2.286987663203794e-06)}},
 'linear': {'medians': {'emissions': np.float64(0.0004282634817614922),
   '_r

In [33]:
import pandas as pd

rows = [{"scorer_name": scorer_name, **contents["medians"]} for scorer_name, contents in aggregated_metrics.items()]
df = pd.DataFrame(rows)
res = df.sort_values(by="energy_consumed", ascending=False)
res

,scorer_name,emissions,_runtime,energy_consumed,gpu_power,cpu_power,gpu_energy,cpu_energy,ram_energy,emissions_rate
5,bert,0.001382,103.911324,0.003133,77.598598,27.0,0.002198,0.000774,1.615405e-04,0.000014
7,ptuning,0.001118,83.454780,0.002535,77.691477,27.0,0.001785,0.000620,1.294621e-04,0.000014
6,lora,0.000863,65.156746,0.001957,76.950353,27.0,0.001372,0.000484,1.009145e-04,0.000013
1,linear,0.000428,73.392776,0.000971,15.459576,27.0,0.000312,0.000545,1.137642e-04,0.000006
3,rerank,0.000270,29.040062,0.000613,44.963938,27.0,0.000355,0.000213,4.436461e-05,0.000010
2,dnnc,0.000122,10.000228,0.000276,74.676038,27.0,0.000192,0.000070,1.454527e-05,0.000013
4,sklearn,0.000073,11.366603,0.000166,21.328199,27.0,0.000074,0.000080,1.663710e-05,0.000007
0,knn,0.000009,1.280654,0.000019,67.440741,27.0,0.000014,0.000004,9.044111e-07,0.000012


In [36]:
res["emissions"] *= 1000
res["energy_consumed"] *= 1000
res["gpu_energy"] *= 1000
res["cpu_energy"] *= 1000
res["ram_energy"] *= 1000
res["emissions_rate"] *= 1000

In [38]:
pd.set_option('display.precision', 3)

In [39]:
print(res.to_string(index=False))

scorer_name  emissions  _runtime  energy_consumed  gpu_power  cpu_power  gpu_energy  cpu_energy  ram_energy  emissions_rate
       bert      1.382   103.911            3.133     77.599       27.0       2.198       0.774   1.615e-01           0.014
    ptuning      1.118    83.455            2.535     77.691       27.0       1.785       0.620   1.295e-01           0.014
       lora      0.863    65.157            1.957     76.950       27.0       1.372       0.484   1.009e-01           0.013
     linear      0.428    73.393            0.971     15.460       27.0       0.312       0.545   1.138e-01           0.006
     rerank      0.270    29.040            0.613     44.964       27.0       0.355       0.213   4.436e-02           0.010
       dnnc      0.122    10.000            0.276     74.676       27.0       0.192       0.070   1.455e-02           0.013
    sklearn      0.073    11.367            0.166     21.328       27.0       0.074       0.080   1.664e-02           0.007
        